# Transformer-LSTM — GAN augmentation comparison (per seed / scene / ratio / GAN)

In [ ]:
import os
import pandas as pd, numpy as np, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import sys; sys.path.insert(0, "../")
from util import *
from models.transformer_lstm import Transformer_LSTM

device = "cuda:1" if torch.cuda.is_available() else "cpu"
print("device:", torch.cuda.get_device_name(device) if "cuda" in device else device)

BASE     = "../CSV_Files"
EVAL_DIR = "../CSV_Files/eval"
GANS     = ["dcgans", "cgans", "wgans"]   # add "timegans" when generated
SEEDS    = [1, 2, 3, 4, 5]
SCENES   = range(1, 7)
RATIOS   = [0.0, 0.5, 1.0, 2.0]
EPOCHS, LR, WD, USE_SCHED, INIT_CONV = 200, 1e-3, 0.0, False, False
MODEL_CLS = Transformer_LSTM
MODEL_NAME = MODEL_CLS.__name__

def train_path(seed, scene, gan, ratio):
    return f"{BASE}/seed{seed}/scene{scene}/train_{gan}_augmented_ratio{ratio}.csv"

RESULTS_CSV = f"{MODEL_NAME.lower()}_gan_compare_raw.csv"


In [ ]:
def to_tensors(arr):
    X = torch.from_numpy(arr[:, 1:14].astype("float32")).unsqueeze(1)
    y = torch.from_numpy(arr[:, -1].astype("int64"))
    return X, y

def tensors_from_arrays(train_arr, val_arr, test_arr):
    X_tr_n, X_va_n, X_te_n, _, _ = preprocess_scenario(train_arr, val_arr, test_arr)
    return to_tensors(X_tr_n), to_tensors(X_va_n), to_tensors(X_te_n)

def make_model(model_cls, device, seed=0, init_conv=True):
    torch.manual_seed(seed)
    model = model_cls(n_classes=8).to(device)
    for m in model.modules():
        if init_conv and isinstance(m, nn.Conv1d):
            nn.init.xavier_uniform_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight); nn.init.zeros_(m.bias)
    return model

def make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, batch_size=50, seed=0):
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(X_tr), generator=g)
    tl = DataLoader(TensorDataset(X_tr[perm], y_tr[perm]), batch_size=batch_size, shuffle=False)
    vl = DataLoader(TensorDataset(X_va, y_va), batch_size=batch_size)
    te = DataLoader(TensorDataset(X_te, y_te), batch_size=batch_size)
    return tl, vl, te

def train_one_epoch(model, loader, crit, opt, device):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()

@torch.no_grad()
def predict_all(model, loader, device):
    model.eval(); yt, yp = [], []
    for xb, yb in loader:
        yp.append(model(xb.to(device)).argmax(1).cpu().numpy()); yt.append(yb.numpy())
    return np.concatenate(yt), np.concatenate(yp)

def run_from_arrays(train_arr, val_arr, test_arr, device, seed=0):
    (X_tr,y_tr),(X_va,y_va),(X_te,y_te) = tensors_from_arrays(train_arr, val_arr, test_arr)
    tl, vl, te = make_loaders(X_tr, y_tr, X_va, y_va, X_te, y_te, 50, seed)
    model = make_model(MODEL_CLS, device, seed=seed, init_conv=INIT_CONV)
    crit = nn.CrossEntropyLoss()
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    sch = optim.lr_scheduler.StepLR(opt, step_size=20, gamma=0.5) if USE_SCHED else None
    for _ in range(EPOCHS):
        train_one_epoch(model, tl, crit, opt, device)
        if sch is not None: sch.step()
    yt, yp = predict_all(model, te, device)
    p, r, f, _ = precision_recall_fscore_support(yt, yp, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(yt, yp), "precision": p, "recall": r, "f1": f, "n_train": len(X_tr)}


In [ ]:
val_arr  = pd.read_csv(f"{EVAL_DIR}/val.csv").to_numpy()
test_arr = pd.read_csv(f"{EVAL_DIR}/test.csv").to_numpy()

done = set()
if os.path.exists(RESULTS_CSV):
    prev = pd.read_csv(RESULTS_CSV)
    done = set(zip(prev.gan, prev.seed.astype(int), prev.scene.astype(int), prev.ratio.astype(float)))

cols = ["gan","seed","scene","ratio","accuracy","precision","recall","f1","n_train"]
for gan in GANS:
    for seed in SEEDS:
        for scene in SCENES:
            for ratio in RATIOS:
                key = (gan, seed, scene, float(ratio))
                if key in done: continue
                path = train_path(seed, scene, gan, ratio)
                if not os.path.exists(path):
                    print(f"MISSING {path}"); continue
                train_arr = pd.read_csv(path).to_numpy()
                res = run_from_arrays(train_arr, val_arr, test_arr, device, seed=seed)
                row = {"gan": gan, "seed": seed, "scene": scene, "ratio": ratio, **res}
                pd.DataFrame([row])[cols].to_csv(RESULTS_CSV, mode="a",
                    header=not os.path.exists(RESULTS_CSV), index=False)
                done.add(key)
                print(f"{gan} seed{seed} scene{scene} ratio{ratio}: F1={res['f1']:.4f}")
print("done")


In [ ]:
df = pd.read_csv(RESULTS_CSV)
agg = df.groupby(["gan","scene","ratio"])["f1"].agg(["mean","std"]).reset_index()
agg.to_csv(f"{MODEL_NAME.lower()}_gan_compare_agg.csv", index=False)
base = agg[agg.ratio==0.0][["gan","scene","mean"]].rename(columns={"mean":"base"})
cmp = agg.merge(base, on=["gan","scene"]); cmp["delta_vs_base"] = cmp["mean"] - cmp["base"]
cmp.to_csv(f"{MODEL_NAME.lower()}_gan_compare_delta.csv", index=False)
aug = cmp[cmp.ratio>0]
best = (aug.loc[aug.groupby(["gan","scene"])["mean"].idxmax()]
           [["gan","scene","ratio","mean","delta_vs_base"]]
           .rename(columns={"ratio":"best_ratio","mean":"best_f1"}))
print("Best augmented F1 per GAN x scene (vs baseline):")
print(best.to_string(index=False))
